# Predictia performantei ofensive a jucatorilor

Acest notebook pregateste datele jucator–meci si compara regresia logistica cu padurea aleatoare. Tinta este **cel putin un gol sau assist in meci**.

Pentru a evita scurgerea de informatii, modelul foloseste doar date din **meciurile anterioare** ale jucatorului. Minutele, golurile, assisturile, suturile si cartonasele devin caracteristici calculate pe ultimele cinci aparitii; golurile si assisturile din meciul curent definesc doar tinta.


## 1. Date de intrare

Ruleaza mai intai `extraction/data_scraper.ipynb`. Notebookul combina automat exporturile FBref si StatsBomb. Pastreaza sursa fiecarui rand, iar istoricul jucatorului este calculat separat in fiecare sursa.


In [1]:
from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd

CWD = Path.cwd().resolve()
if CWD.name == "notebooks" and (CWD.parent / "README.md").exists():
    PROJECT_ROOT = CWD.parent
elif (CWD / "notebooks").is_dir() and (CWD / "README.md").exists():
    PROJECT_ROOT = CWD
else:
    candidates = [p for p in (CWD, *CWD.parents) if (p / "notebooks").is_dir() and (p / "README.md").exists()]
    if not candidates:
        raise RuntimeError("Deschide Jupyter din IAPD_project/ sau din directorul notebooks/.")
    PROJECT_ROOT = candidates[0]

DATA_ROOT = PROJECT_ROOT / "data"
MODEL_SOURCES = ["fbref", "statsbomb"]  # Foloseste ambele surse.
INPUT_FILES = {
    "fbref": DATA_ROOT / "raw" / "fbref" / "fbref_player_match_stats.csv",
    "statsbomb": DATA_ROOT / "raw" / "statsbomb" / "statsbomb_player_match_stats.csv",
}
if not MODEL_SOURCES or any(source not in INPUT_FILES for source in MODEL_SOURCES):
    raise ValueError(f"Surse invalide: {MODEL_SOURCES}")
OUTPUT_DIR = DATA_ROOT / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Asociaza fiecare camp cu numele coloanei din setul de date ales.
# Ambele exporturi data_scraper contin aceste campuri comune.
COLUMN_MAP = {
    "player": "player",
    "match_date": "date",
    "minutes": "Min",
    "goals": "Gls",
    "assists": "Ast",
    "shots": "Sh",
    "yellow_cards": "CrdY",
    "red_cards": "CrdR",
    # Optional: "match_id": "game_id",
}

In [2]:
frames = []
for source in MODEL_SOURCES:
    input_file = INPUT_FILES[source]
    if not input_file.exists():
        raise FileNotFoundError(
            f"Fisierul {input_file} lipseste. Ruleaza mai intai extraction/data_scraper.ipynb."
        )
    source_frame = pd.read_csv(input_file)
    source_frame["source"] = source
    frames.append(source_frame)

raw = pd.concat(frames, ignore_index=True, sort=False)
MIN_DATA_ROWS = 1_000
if len(raw) < MIN_DATA_ROWS:
    raise ValueError(f"Criteriul setului de date nu este indeplinit: {len(raw):,} randuri; sunt necesare cel putin {MIN_DATA_ROWS:,}.")
print(f"Incarcate {len(raw):,} randuri din sursele {', '.join(MODEL_SOURCES)}")
print("Randuri pe sursa:")
display(raw.groupby("source").size().rename("randuri").to_frame())
raw.head()


Incarcate 1,995 randuri si 31 coloane
Available coloane: ['source', 'competition', 'season', 'match_id', 'date', 'player_id', 'player', 'team_id', 'team', 'opponent', 'venue', 'position', 'Min', 'Gls', 'Ast', 'Sh', 'SoT', 'xG', 'CrdY', 'CrdR', 'Passes', 'PassesCompleted', 'KeyPasses', 'Crosses', 'Carries', 'Pressures', 'Interceptions', 'Recoveries', 'Blocks', 'Clearances', 'FoulsCommitted']


,source,competition,season,match_id,date,player_id,player,team_id,team,opponent,...,PassesCompleted,KeyPasses,Crosses,Carries,Pressures,Interceptions,Recoveries,Blocks,Clearances,FoulsCommitted
0,statsbomb,FIFA World Cup,2022,3857276,2022-12-01,3625,Sofiane Boufal,788,Morocco,Canada,...,15,2,0,22,24,1,2,2,0,5
1,statsbomb,FIFA World Cup,2022,3857276,2022-12-01,3634,Abdelhamid Sabiri,788,Morocco,Canada,...,21,0,0,20,12,2,3,1,1,0
2,statsbomb,FIFA World Cup,2022,3857276,2022-12-01,5219,Romain Saiss,788,Morocco,Canada,...,30,0,0,30,6,1,3,0,5,0
3,statsbomb,FIFA World Cup,2022,3857276,2022-12-01,5234,Sofyan Amrabat,788,Morocco,Canada,...,43,0,0,39,22,2,4,0,3,3
4,statsbomb,FIFA World Cup,2022,3857276,2022-12-01,5237,Hakim Ziyech,788,Morocco,Canada,...,26,0,4,36,18,1,4,2,1,1


## 2. Standardizarea coloanelor

Folosim aceleasi campuri comune din ambele surse. Coloana `source` ramane in date pentru a distinge competitia si furnizorul. Caracteristicile istorice se calculeaza separat pentru fiecare jucator si sursa, ca sa nu amestecam meciurile de club cu cele internationale.


In [3]:
required = ["player", "match_date", "minutes", "goals", "assists", "shots", "yellow_cards", "red_cards"]
missing_mapping = [key for key in required if COLUMN_MAP.get(key) not in raw.columns]
if missing_mapping:
    raise KeyError(
        "Nu au fost gasite coloanele mapate: " + ", ".join(f"{key} -> {COLUMN_MAP.get(key)!r}" for key in missing_mapping)
        + ". Actualizeaza COLUMN_MAP cu antetele exacte de mai sus."
    )

rename = {source: canonical for canonical, source in COLUMN_MAP.items() if source in raw.columns}
data = raw.rename(columns=rename).copy()
if "match_id" in COLUMN_MAP and COLUMN_MAP["match_id"] in raw.columns:
    data = data.rename(columns={COLUMN_MAP["match_id"]: "match_id"})

data["match_date"] = pd.to_datetime(data["match_date"], format="mixed", errors="coerce", utc=True)
numeric = ["minutes", "goals", "assists", "shots", "yellow_cards", "red_cards"]
for col in numeric:
    data[col] = pd.to_numeric(data[col], errors="coerce")
data["player"] = data["player"].astype("string").str.strip()
data = data.dropna(subset=["player", "match_date"]).copy()
data = data[data["minutes"].fillna(0) > 0].copy()
data["source_is_statsbomb"] = (data["source"] == "statsbomb").astype(int)
data = data.sort_values(["source", "player", "match_date"]).reset_index(drop=True)

print(f"Randuri jucator-meci utilizabile: {len(data):,}")
print("Valori lipsa pe camp:")
display(data[required].isna().sum().to_frame("missing_rows"))

Randuri jucator-meci utilizabile: 1,995
Valori lipsa pe camp:


,missing_rows
player,0
match_date,0
minutes,0
goals,0
assists,0
shots,0
yellow_cards,0
red_cards,0


## 3. Definirea tintei si crearea caracteristicilor pre-meci

Performanta ofensiva ridicata inseamna cel putin un gol sau assist in meciul curent. Caracteristicile folosesc ultimele cinci aparitii anterioare ale jucatorului in aceeasi sursa. Indicatorul sursei ajuta modelul sa tina cont de diferentele dintre statisticile FBref si StatsBomb.


In [4]:
data = data.dropna(subset=["goals", "assists"]).copy()
data["goal_contributions"] = data["goals"] + data["assists"]
data["high_offensive_performance"] = (data["goal_contributions"] >= 1).astype(int)

past_fields = ["minutes", "goals", "assists", "shots", "yellow_cards", "red_cards"]
for field in past_fields:
    data[f"recent5_mean_{field}"] = (
        data.groupby(["source", "player"], sort=False)[field]
        .transform(lambda series: series.shift(1).rolling(window=5, min_periods=1).mean())
    )
    data[f"recent5_sum_{field}"] = (
        data.groupby(["source", "player"], sort=False)[field]
        .transform(lambda series: series.shift(1).rolling(window=5, min_periods=1).sum())
    )

feature_cols = [col for col in data.columns if col.startswith("recent5_")] + ["source_is_statsbomb"]
model_data = data.dropna(subset=feature_cols + ["high_offensive_performance"]).copy()
print("Distributia tintei (0 = fara contributie la gol, 1 = cel putin una):")
display(model_data["high_offensive_performance"].value_counts(normalize=True).rename("share").to_frame())
if len(model_data) < MIN_DATA_ROWS:
    raise ValueError(
        f"Criteriul pentru setul de modelare nu este indeplinit: {len(model_data):,} randuri jucator-meci utilizabile; "
        f"sunt necesare cel putin {MIN_DATA_ROWS:,}."
    )
if len(feature_cols) < 10:
    raise ValueError(f"Criteriul caracteristicilor modelului nu este indeplinit: {len(feature_cols)} engineered features; sunt necesare cel putin 10.")
print(f"OK — setul de modelare are {len(model_data):,} randuri utilizabile si {len(feature_cols):,} caracteristici pre-meci")

Distributia tintei (0 = fara contributie la gol, 1 = cel putin una):


,share
high_offensive_performance,
0,0.858447
1,0.141553


OK — setul de modelare are 1,314 usable randuri si 12 caracteristici pre-meci


## 4. Impartirea cronologica antrenare/test

Pentru fiecare sursa, cele mai recente 20% dintre datele distincte sunt pastrate pentru test. Astfel, evaluarea verifica predictia unor meciuri ulterioare in ambele seturi de date, fara sa amestece aceeasi zi intre antrenare si test.


In [5]:
train_parts = []
test_parts = []
for source, source_data in model_data.groupby("source", sort=False):
    source_dates = source_data["match_date"].drop_duplicates().sort_values().reset_index(drop=True)
    split_index = int(len(source_dates) * 0.8)
    if split_index <= 0 or split_index >= len(source_dates):
        raise ValueError(f"Nu sunt suficiente date distincte pentru impartirea cronologica a sursei {source}.")
    split_date = source_dates.iloc[split_index]
    train_parts.append(source_data[source_data["match_date"] < split_date].copy())
    test_parts.append(source_data[source_data["match_date"] >= split_date].copy())

train = pd.concat(train_parts, ignore_index=True)
test = pd.concat(test_parts, ignore_index=True)
X_train, X_test = train[feature_cols], test[feature_cols]
y_train = train["high_offensive_performance"]
y_test = test["high_offensive_performance"]
if y_train.nunique() < 2 or y_test.nunique() < 2:
    raise ValueError("Impartirea cronologica nu contine ambele clase in antrenare si test.")
print(f"Randuri de antrenare: {len(train):,}")
print(f"Randuri de test: {len(test):,}")
print("Randuri de test pe sursa:")
display(test.groupby("source").size().rename("randuri").to_frame())


Randuri de antrenare: 1,196 through 2022-12-10
Randuri de test: 118 from 2022-12-13


## 5. Antrenarea si compararea modelelor

Ambele modele completeaza valorile lipsa din caracteristici cu mediana. Regresia logistica isi si scaleaza intrarile. Ponderarea claselor ajuta cand contributiile la goluri sunt rare. Comparatia raporteaza precizia, rata de identificare, scorul F1, ROC AUC si media preciziei (PR AUC).


In [6]:
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
)

models = {
    "Regresie logistica": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(class_weight="balanced", max_iter=2000, random_state=42)),
    ]),
    "Padure aleatoare": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(
            n_estimators=400, min_samples_leaf=3, class_weight="balanced",
            random_state=42, n_jobs=-1,
        )),
    ]),
}

results = []
predictions = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    predicted = model.predict(X_test)
    probability = model.predict_proba(X_test)[:, 1]
    predictions[name] = (predicted, probability)
    results.append({
        "model": name,
        "accuracy": accuracy_score(y_test, predicted),
        "precision": precision_score(y_test, predicted, zero_division=0),
        "recall": recall_score(y_test, predicted, zero_division=0),
        "f1": f1_score(y_test, predicted, zero_division=0),
        "roc_auc": roc_auc_score(y_test, probability),
        "average_precision": average_precision_score(y_test, probability),
    })

comparison = pd.DataFrame(results).set_index("model").sort_values("average_precision", ascending=False)
display(comparison.round(3))

,accuracy,precision,recall,f1,roc_auc,average_precision
model,,,,,,
Regresie logistica,0.585,0.218,0.667,0.329,0.680,0.416
Padure aleatoare,0.619,0.171,0.389,0.237,0.582,0.205


## 6. Evaluarea modelelor si salvarea rezultatelor

Pe langa scorul general, verifica rezultatele separat pentru FBref si StatsBomb. Combinarea surselor nu garanteaza automat o predictie mai buna; diferentele de competitie si definitii statistice pot influenta scorurile.


In [7]:
best_name = comparison.index[0]
best_model = models[best_name]
best_pred, best_probability = predictions[best_name]
print("Selectat dupa media preciziei:", best_name)
print("Matrice de confuzie (randuri = real [0, 1], coloane = prezis [0, 1]):")
print(confusion_matrix(y_test, best_pred))

comparison.to_csv(OUTPUT_DIR / "model_comparison.csv", encoding="utf-8-sig")
test_results = test[["source", "player", "match_date", "goals", "assists", "high_offensive_performance"]].copy()
test_results["predicted_high_performance"] = best_pred
test_results["predicted_probability"] = best_probability
test_results.to_csv(OUTPUT_DIR / "test_predictions.csv", index=False, encoding="utf-8-sig")
source_scores = []
for source, source_test in test.groupby("source"):
    positions = source_test.index
    source_pred = pd.Series(best_pred, index=test.index).loc[positions]
    source_prob = pd.Series(best_probability, index=test.index).loc[positions]
    source_scores.append({
        "source": source,
        "rows": len(source_test),
        "precision": precision_score(source_test["high_offensive_performance"], source_pred, zero_division=0),
        "recall": recall_score(source_test["high_offensive_performance"], source_pred, zero_division=0),
        "f1": f1_score(source_test["high_offensive_performance"], source_pred, zero_division=0),
        "average_precision": average_precision_score(source_test["high_offensive_performance"], source_prob),
    })
source_comparison = pd.DataFrame(source_scores).set_index("source")
display(source_comparison.round(3))
source_comparison.to_csv(OUTPUT_DIR / "source_metrics.csv", encoding="utf-8-sig")
print("Salvat:", OUTPUT_DIR / "model_comparison.csv")
print("Salvat:", OUTPUT_DIR / "test_predictions.csv")
print("Salvat:", OUTPUT_DIR / "source_metrics.csv")

Selectat dupa media preciziei: Regresie logistica
Confusion matrix (rows = actual [0, 1], coloane = predicted [0, 1]):
[[57 43]
 [ 6 12]]
Salvat: /Users/laur/Master Data Science/sem1/IAPD/IAPD_project/notebooks/data/processed/model_comparison.csv
Salvat: /Users/laur/Master Data Science/sem1/IAPD/IAPD_project/notebooks/data/processed/test_predictions.csv


## Interpretare si limitari

- Contributia la gol este o tinta simpla, dar nu o definitie universala a performantei ofensive.
- Modelul foloseste forma recenta si sursa datelor; nu include adversarul, accidentarile sau minutele estimate inaintea meciului.
- FBref si StatsBomb acopera competitii diferite si pot defini statisticile diferit. De aceea istoricul este separat pe sursa, iar scorurile sunt raportate si separat.
- Mai multe date pot ajuta, dar nu garanteaza acuratete mai buna. Compara metricile per sursa si adauga sezoane pentru a obtine o evaluare mai solida.
